# 自动并行与多GPU基础

本notebook介绍自动并行化机制和多GPU训练的基础概念。

## 学习目标

- 理解框架的自动并行机制
- 掌握多GPU训练的基本原理
- 学习数据并行的实现方法
- 了解参数同步策略

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import time

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. 自动并行化

### 1.1 为什么需要自动并行?

**挑战**:
- 现代CPU有多个核心(8-64个)
- 单个操作通常只用1个核心
- 手动并行化代码复杂且易错

**解决方案**: 深度学习框架自动并行化
- 框架分析计算图
- 识别独立操作
- 自动调度到不同设备

### 1.2 单操作的并行

**CPU操作**:
```python
x = torch.randn(10000, 10000)  # CPU上
y = torch.mm(x, x)  # 自动使用所有CPU核心
```

**GPU操作**:
```python
x = torch.randn(10000, 10000, device='cuda')
y = torch.mm(x, x)  # 使用GPU的所有SM
```

**关键**: 单个操作已经充分利用设备资源!

In [ ]:
# 演示单操作的并行性
def benchmark_single_op():
    """对比CPU和GPU的矩阵乘法"""
    n = 5000
    
    # CPU版本
    x_cpu = torch.randn(n, n)
    start = time.time()
    for _ in range(10):
        y_cpu = torch.mm(x_cpu, x_cpu)
    cpu_time = time.time() - start
    print(f"CPU (所有核心): {cpu_time:.3f}秒")
    
    # GPU版本(如果可用)
    if torch.cuda.is_available():
        x_gpu = torch.randn(n, n, device='cuda')
        torch.cuda.synchronize()  # 预热
        
        start = time.time()
        for _ in range(10):
            y_gpu = torch.mm(x_gpu, x_gpu)
        torch.cuda.synchronize()
        gpu_time = time.time() - start
        print(f"GPU (所有SM): {gpu_time:.3f}秒")
        print(f"\nGPU加速: {cpu_time/gpu_time:.1f}x")
    else:
        print("\n没有GPU,无法对比")

benchmark_single_op()

### 1.3 多设备并行

**真正的并行**: 多个独立操作在不同设备上同时执行

**示例场景**:
```python
# 两个独立计算
x1 = torch.randn(1000, 1000, device='cuda:0')
x2 = torch.randn(1000, 1000, device='cuda:1')

y1 = compute(x1)  # GPU 0上执行
y2 = compute(x2)  # GPU 1上并行执行
```

**依赖关系**:
```python
a = op1()  # 操作1
b = op2()  # 操作2,与op1并行
c = op3(a, b)  # 操作3,依赖a和b,必须等待
```

框架自动跟踪依赖,调度执行!

In [ ]:
# 演示多设备并行
def demo_multi_device_parallel():
    """演示多GPU并行执行"""
    if torch.cuda.device_count() < 2:
        print("需要至少2个GPU来演示多设备并行")
        print("使用单GPU模拟...\n")
        device1 = device2 = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    else:
        device1 = torch.device('cuda:0')
        device2 = torch.device('cuda:1')
    
    n = 4000
    
    # 串行执行(在同一设备上)
    print("=== 串行执行 ===")
    x1 = torch.randn(n, n, device=device1)
    x2 = torch.randn(n, n, device=device1)
    
    start = time.time()
    y1 = torch.mm(x1, x1)  # 计算1
    y2 = torch.mm(x2, x2)  # 计算2
    if torch.cuda.is_available():
        torch.cuda.synchronize(device1)
    serial_time = time.time() - start
    print(f"时间: {serial_time:.4f}秒")
    
    # 并行执行(在不同设备上)
    if torch.cuda.device_count() >= 2:
        print("\n=== 并行执行(2个GPU) ===")
        x1 = torch.randn(n, n, device=device1)
        x2 = torch.randn(n, n, device=device2)
        
        start = time.time()
        y1 = torch.mm(x1, x1)  # GPU 0
        y2 = torch.mm(x2, x2)  # GPU 1,并行!
        torch.cuda.synchronize(device1)
        torch.cuda.synchronize(device2)
        parallel_time = time.time() - start
        print(f"时间: {parallel_time:.4f}秒")
        print(f"\n加速比: {serial_time/parallel_time:.2f}x")
        print("理论加速: 2x (两个GPU并行工作)")

demo_multi_device_parallel()

### 1.4 计算与通信重叠

**关键洞察**: 计算和通信使用不同资源

**资源类型**:
- **计算资源**: CPU核心、GPU SM
- **通信资源**: PCIe总线、网络带宽

**重叠策略**:
```python
# 同时进行计算和数据传输
for i in range(n):
    # 在GPU上计算
    result = compute_on_gpu(data[i])
    
    # 异步传输到CPU(利用PCIe总线)
    result_cpu = result.to('cpu', non_blocking=True)
    
    # 计算和传输并行!
```

**性能提升**:
- 串行: $T_{\text{total}} = T_{\text{compute}} + T_{\text{transfer}}$
- 重叠: $T_{\text{total}} \approx \max(T_{\text{compute}}, T_{\text{transfer}})$

In [ ]:
# 演示计算与通信重叠
def demo_compute_comm_overlap():
    """演示计算与数据传输的重叠"""
    if not torch.cuda.is_available():
        print("需要GPU")
        return
    
    device = torch.device('cuda')
    n_iterations = 20
    size = (2000, 2000)
    
    # 方法1: 串行(计算→等待→传输)
    print("=== 串行执行 ===")
    start = time.time()
    for i in range(n_iterations):
        # 计算
        x = torch.randn(size, device=device)
        y = torch.mm(x, x)
        
        # 同步(等待计算完成)
        torch.cuda.synchronize()
        
        # 传输
        y_cpu = y.to('cpu')
    serial_time = time.time() - start
    print(f"时间: {serial_time:.4f}秒")
    
    # 方法2: 重叠(计算的同时异步传输)
    print("\n=== 重叠执行 ===")
    start = time.time()
    for i in range(n_iterations):
        # 计算
        x = torch.randn(size, device=device)
        y = torch.mm(x, x)
        
        # 异步传输(不等待计算)
        y_cpu = y.to('cpu', non_blocking=True)
    
    # 最后统一同步
    torch.cuda.synchronize()
    overlap_time = time.time() - start
    print(f"时间: {overlap_time:.4f}秒")
    
    print(f"\n性能提升: {serial_time/overlap_time:.2f}x")
    print("原理: 当前批次传输时,下一批次已在计算")

demo_compute_comm_overlap()

## 2. 多GPU训练策略

### 2.1 三种并行方式

#### 方式1: 网络并行 ❌ (不推荐)

**思路**: 不同层在不同GPU上
```
GPU 0: Layer 1-2
GPU 1: Layer 3-4
GPU 2: Layer 5-6
```

**问题**:
- 层间频繁通信(每层输出传给下层)
- GPU利用率低(上一层未完成,下一层空闲)
- 负载不均衡(层计算量差异大)

#### 方式2: 层内并行 ❌ (不推荐)

**思路**: 每层的通道/神经元分配到不同GPU
```
Conv 64通道:
  GPU 0: 通道 0-15
  GPU 1: 通道 16-31
  GPU 2: 通道 32-47
  GPU 3: 通道 48-63
```

**问题**:
- 每层都需要同步(All-to-All通信)
- 通信开销巨大
- 实现复杂

#### 方式3: 数据并行 ✅ (强烈推荐)

**思路**: 数据分批,每个GPU处理不同数据
```
Batch 256:
  GPU 0: 样本 0-63
  GPU 1: 样本 64-127
  GPU 2: 样本 128-191
  GPU 3: 样本 192-255
```

**优点**:
- ✅ 简单: 每GPU独立前向/反向传播
- ✅ 高效: 仅在梯度聚合时通信
- ✅ 可扩展: 容易扩展到多台机器
- ✅ 通用: 适用于任何模型

### 2.2 数据并行工作流

**假设**: $k$ 个GPU,批大小 $b$

**步骤**:

1. **数据分割**: 将批次 $b$ 分成 $k$ 份,每份 $b/k$
   ```
   GPU i 获得样本: data[i*b/k : (i+1)*b/k]
   ```

2. **并行前向**: 每个GPU独立计算
   ```python
   # GPU i上
   output_i = model(data_i)
   loss_i = criterion(output_i, target_i)
   ```

3. **并行反向**: 每个GPU计算局部梯度
   ```python
   loss_i.backward()  # 得到 grad_i
   ```

4. **梯度聚合**: 汇总所有GPU的梯度
   ```python
   grad_total = sum([grad_0, grad_1, ..., grad_{k-1}]) / k
   ```

5. **参数更新**: 每个GPU用聚合梯度更新
   ```python
   optimizer.step()  # 使用grad_total
   ```

6. **同步模型**: 确保所有GPU参数一致
   ```python
   # 通常框架自动处理
   broadcast_parameters()
   ```

## 3. 数据并行实现

### 3.1 手动实现数据并行

理解底层机制很重要!

In [ ]:
# 手动实现数据并行
def split_batch(data, target, devices):
    """将批次分割到多个设备"""
    split_size = data.size(0) // len(devices)
    data_splits = []
    target_splits = []
    
    for i, device in enumerate(devices):
        start = i * split_size
        end = start + split_size if i < len(devices) - 1 else data.size(0)
        
        data_splits.append(data[start:end].to(device))
        target_splits.append(target[start:end].to(device))
    
    return data_splits, target_splits

def allreduce_gradients(models):
    """聚合多个模型的梯度"""
    # 对每个参数,求所有GPU上梯度的平均
    for params in zip(*[model.parameters() for model in models]):
        # 聚合梯度到第一个GPU
        grads = [p.grad.data for p in params]
        avg_grad = sum(g.to(grads[0].device) for g in grads) / len(grads)
        
        # 广播回所有GPU
        for p in params:
            p.grad.data = avg_grad.to(p.device)

# 简单模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

print("数据并行核心组件:")
print("1. split_batch: 分割数据到各GPU")
print("2. allreduce_gradients: 聚合梯度")
print("3. 每个GPU独立前向/反向传播")

### 3.2 PyTorch DataParallel

**最简单的多GPU方案**:

In [ ]:
# 使用DataParallel
def demo_data_parallel():
    """演示nn.DataParallel的使用"""
    model = SimpleNet()
    
    if torch.cuda.device_count() > 1:
        print(f"使用 {torch.cuda.device_count()} 个GPU训练!")
        # 包装模型
        model = nn.DataParallel(model)
    elif torch.cuda.is_available():
        print("只有1个GPU,使用单GPU训练")
    else:
        print("没有GPU,使用CPU训练")
    
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # 训练示例
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    # 模拟一个批次
    batch_size = 64 * torch.cuda.device_count() if torch.cuda.is_available() else 64
    data = torch.randn(batch_size, 1, 28, 28).to(device)
    target = torch.randint(0, 10, (batch_size,)).to(device)
    
    # 前向传播
    output = model(data)  # DataParallel自动分割数据!
    loss = criterion(output, target)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()  # DataParallel自动聚合梯度!
    optimizer.step()
    
    print(f"\n训练完成! 损失: {loss.item():.4f}")
    print("\nDataParallel优点:")
    print("  ✅ 使用简单,只需1行代码")
    print("  ✅ 自动数据分割")
    print("  ✅ 自动梯度聚合")
    print("\nDataParallel缺点:")
    print("  ❌ 只支持单机多GPU")
    print("  ❌ GPU 0负载更重(聚合点)")
    print("  ❌ Python GIL限制")

demo_data_parallel()

### 3.3 DistributedDataParallel (推荐)

**更高效的多GPU/多机方案**:

**优势**:
- ✅ 每个进程独立Python解释器(无GIL)
- ✅ 更均衡的GPU负载
- ✅ 支持多机训练
- ✅ 更快的梯度同步(All-Reduce)

**基本用法**:
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# 初始化进程组
dist.init_process_group(backend='nccl')

# 为每个进程设置设备
local_rank = int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(local_rank)

# 包装模型
model = model.to(local_rank)
model = DDP(model, device_ids=[local_rank])

# 训练(每个进程独立运行)
for data, target in dataloader:
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
```

**启动命令**:
```bash
# 单机4卡
torchrun --nproc_per_node=4 train.py

# 多机(2台,每台4卡)
# 机器0:
torchrun --nproc_per_node=4 --nnodes=2 --node_rank=0 \
         --master_addr=192.168.1.1 --master_port=29500 train.py
# 机器1:
torchrun --nproc_per_node=4 --nnodes=2 --node_rank=1 \
         --master_addr=192.168.1.1 --master_port=29500 train.py
```

## 4. 参数同步策略

### 4.1 All-Reduce vs Parameter Server

#### All-Reduce (推荐)

**思路**: 所有GPU直接通信,无中心节点

**算法**: Ring All-Reduce
```
步骤1: 将梯度分成N块(N=GPU数)
步骤2: 每个GPU向邻居发送1块
步骤3: 经过N-1轮后,每个GPU有1块的完整和
步骤4: 再传N-1轮,所有GPU得到完整聚合梯度
```

**优点**:
- 带宽利用最优: $O(1)$ 复杂度(不随GPU数增长)
- 无单点瓶颈
- 负载均衡

**PyTorch使用**: NCCL backend自动使用Ring All-Reduce

#### Parameter Server

**思路**: 中心服务器收集梯度,更新参数,再分发

**流程**:
```
1. Workers → Server: 发送梯度
2. Server: 聚合梯度,更新参数
3. Server → Workers: 广播新参数
```

**问题**:
- 服务器带宽瓶颈: $O(N)$ 复杂度
- 单点故障

**解决**: 多参数服务器
- 参数分片到多个服务器
- 每个服务器负责部分参数
- 降低单点压力

### 4.2 同步vs异步训练

#### 同步训练 (Synchronous)

**特点**: 所有GPU同步更新
```python
# 伪代码
for epoch in epochs:
    # 所有GPU并行前向/反向
    gradients = [gpu_i.backward() for i in gpus]
    
    # 等待所有GPU完成
    barrier()
    
    # 聚合梯度
    avg_grad = mean(gradients)
    
    # 所有GPU同步更新
    for gpu in gpus:
        gpu.update(avg_grad)
```

**优点**:
- ✅ 收敛稳定(等价于大批量训练)
- ✅ 易于调试
- ✅ 确定性结果

**缺点**:
- ❌ 慢GPU拖慢整体(木桶效应)
- ❌ 需要等待所有GPU

#### 异步训练 (Asynchronous)

**特点**: GPU独立更新,不等待
```python
# 每个GPU独立运行
for epoch in epochs:
    grad = gpu.backward()
    
    # 发送梯度到服务器(不等待)
    server.push(grad)
    
    # 立即拉取最新参数(可能不是用自己梯度更新的)
    params = server.pull()
    gpu.update(params)
```

**优点**:
- ✅ 快GPU不等慢GPU
- ✅ 更高吞吐量

**缺点**:
- ❌ 收敛不稳定(梯度过时)
- ❌ 需要调参(学习率等)
- ❌ 不确定性

**实践**: 深度学习通常用**同步训练**!

## 5. 多GPU训练最佳实践

### 5.1 批大小调整

**线性缩放规则**:
```
k个GPU → 批大小 × k
```

**原因**: 保持每GPU的工作量一致

**示例**:
```python
# 单GPU: batch_size=64
# 4个GPU: batch_size=256 (64×4)
# 8个GPU: batch_size=512 (64×8)
```

### 5.2 学习率调整

**线性缩放规则** (常用):
```
批大小 × k → 学习率 × k
```

**理由**: 大批次梯度更稳定,可用更大学习率

**例外**: 可能需要warmup
```python
# 前5个epoch线性增加学习率
for epoch in range(5):
    lr = base_lr * (epoch + 1) / 5
```

### 5.3 BatchNorm处理

**问题**: 每GPU的batch较小,BN统计不准

**解决方案**:
1. **SyncBatchNorm**: 跨GPU同步统计
   ```python
   model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
   ```

2. **GroupNorm**: 不依赖batch
   ```python
   nn.GroupNorm(num_groups=32, num_channels=256)
   ```

3. **LayerNorm**: 按层归一化
   ```python
   nn.LayerNorm(normalized_shape)
   ```

### 5.4 性能优化checklist

**数据加载**:
- [ ] `pin_memory=True` in DataLoader
- [ ] 足够的`num_workers`(通常=GPU数×4)
- [ ] 预取数据(prefetch_factor)

**通信优化**:
- [ ] 使用NCCL backend (GPU)
- [ ] 梯度累积减少通信
- [ ] 梯度压缩(FP16)

**内存优化**:
- [ ] 梯度检查点(gradient checkpointing)
- [ ] 混合精度训练(AMP)
- [ ] 每GPU批大小最大化

**调试**:
- [ ] 先在单GPU上调通
- [ ] 验证多GPU loss一致性
- [ ] 监控GPU利用率
- [ ] 检查通信开销

In [ ]:
# 完整的多GPU训练模板
class MultiGPUTrainer:
    """多GPU训练最佳实践模板"""
    
    def __init__(self, model, device_ids=None):
        # 自动检测GPU
        if device_ids is None:
            device_ids = list(range(torch.cuda.device_count()))
        
        self.device_ids = device_ids
        self.num_gpus = len(device_ids)
        
        # 设置主设备
        self.device = torch.device(f'cuda:{device_ids[0]}' if device_ids else 'cpu')
        
        # 包装模型
        model = model.to(self.device)
        if self.num_gpus > 1:
            model = nn.DataParallel(model, device_ids=device_ids)
        self.model = model
        
        print(f"初始化训练器: {self.num_gpus} GPU(s)")
    
    def get_scaled_lr(self, base_lr):
        """根据GPU数量缩放学习率"""
        return base_lr * self.num_gpus
    
    def get_scaled_batch_size(self, base_batch_size):
        """根据GPU数量缩放批大小"""
        return base_batch_size * self.num_gpus
    
    def train_epoch(self, dataloader, optimizer, criterion):
        """训练一个epoch"""
        self.model.train()
        total_loss = 0
        
        for batch_idx, (data, target) in enumerate(dataloader):
            # 数据移到设备
            data = data.to(self.device, non_blocking=True)
            target = target.to(self.device, non_blocking=True)
            
            # 前向传播
            optimizer.zero_grad()
            output = self.model(data)
            loss = criterion(output, target)
            
            # 反向传播
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(dataloader)

# 使用示例
print("\n=== 多GPU训练模板 ===")
print("\n关键配置:")
print("1. 批大小 = base_batch × GPU数")
print("2. 学习率 = base_lr × GPU数")
print("3. DataLoader: pin_memory=True, num_workers充足")
print("4. 数据传输: non_blocking=True")
print("\n推荐组合:")
print("  单GPU: batch=64, lr=0.1")
print("  4-GPU: batch=256, lr=0.4")
print("  8-GPU: batch=512, lr=0.8")

## 6. 小结

### 核心要点

1. **自动并行**
   - 单操作自动用满设备资源
   - 框架自动调度独立操作
   - 计算与通信可重叠

2. **多GPU策略**
   - **数据并行**: 首选,简单高效
   - 网络并行: 不推荐(通信多)
   - 层内并行: 不推荐(复杂)

3. **数据并行工作流**
   - 数据分割 → 并行计算 → 梯度聚合 → 参数更新
   - PyTorch: `nn.DataParallel` (简单) 或 `DDP` (高效)

4. **同步策略**
   - **All-Reduce**: 无中心,高效,推荐
   - Parameter Server: 有瓶颈,可多服务器
   - **同步训练**: 稳定,常用
   - 异步训练: 快但不稳

5. **最佳实践**
   - 批大小 × GPU数
   - 学习率可能需要调整(通常×GPU数)
   - 使用SyncBatchNorm或GroupNorm
   - pin_memory + non_blocking传输

### 技术选择指南

| 场景 | 推荐方案 |
|------|----------|
| 单机多GPU | `nn.DataParallel` (简单) |
| 单机多GPU (高性能) | `DistributedDataParallel` |
| 多机多GPU | `DistributedDataParallel` + NCCL |
| 模型太大单GPU装不下 | 模型并行 (Megatron, DeepSpeed) |
| 超大模型(GPT-3级别) | 3D并行 (数据+模型+流水线) |

### 性能调优步骤

1. **建立基线**: 单GPU性能
2. **扩展到多GPU**: 验证加速比
3. **识别瓶颈**: Profiling (计算 vs 通信)
4. **优化通信**: 梯度压缩,异步传输
5. **优化计算**: 混合精度,算子融合
6. **监控**: GPU利用率,通信时间

## 练习

1. **自动并行实验**: 创建两个独立计算,验证它们在多GPU上是否并行执行。

2. **数据并行实现**: 手动实现一个简单的数据并行训练循环(不用DataParallel)。

3. **性能对比**: 在1/2/4个GPU上训练同一模型,测量训练时间和加速比。

4. **批大小实验**: 固定总批大小(如256),在不同GPU数上训练,观察收敛差异。

5. **通信分析**: 使用PyTorch Profiler分析多GPU训练的通信开销占比。

6. **DDP实践**: 将DataParallel代码改写为DistributedDataParallel,对比性能。